# TwoWayTransformer Attention Map Visualization

This notebook provides comprehensive visualization of attention maps between prompts and image features in SAM's TwoWayTransformer (MaskDecoder).

## Overview
- Visualizes attention patterns in TwoWayTransformer layers
- Shows how prompts influence attention to image regions
- Provides multiple visualization modes and statistical analysis


In [ ]:
# Imports
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from collections import defaultdict
from typing import Union, Tuple, Optional, Callable
from  
from functools import partial
import cv2

# SAM imports
from segment_anything import build_sam, build_sam_hq, sam_hq_model_registry, SamPredictor

print("Imports completed successfully!")


ImportError: cannot import name 'TinyViT' from 'segment_anything.modeling' (/home/chauht2/SAM_Quantization/.venv/lib/python3.10/site-packages/segment_anything/modeling/__init__.py)

In [10]:
# Utility Functions
def to_numpy(x: torch.Tensor):
    return x.detach().cpu().numpy()

def show_points(coords, labels, ax, marker_size=200):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    
def show_mask_image(mask, ax, random_color=False, borders=True):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask = mask.astype(np.uint8)
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    if borders:
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
        mask_image = cv2.drawContours(mask_image, contours, -1, (1, 1, 1, 0.5), thickness=2)
    ax.imshow(mask_image)

print("Utility functions loaded!")


Utility functions loaded!


In [11]:
# Enhanced TwoWayTransformer Attention Hook
class EnhancedTwoWayTransformerHook:
    """
    Enhanced hook for capturing attention weights from TwoWayTransformer
    This modifies the forward pass to capture attention weights
    """
    def __init__(self):
        self.attention_weights = {}
        self.hooks = []
        
    def hook_two_way_transformer(self, model):
        """Hook TwoWayTransformer to capture attention weights"""
        print("Hooking TwoWayTransformer for attention capture...")
        
        # Find TwoWayTransformer modules in mask_decoder
        two_way_transformers = []
        for name, module in model.named_modules():
            if 'mask_decoder' in name and 'TwoWayTransformer' in str(type(module)):
                two_way_transformers.append((name, module))
        
        print(f"Found {len(two_way_transformers)} TwoWayTransformer modules")
        
        for name, module in two_way_transformers:
            # Hook each attention layer within the TwoWayTransformer
            self._hook_attention_layers(name, module)
        
        print(f"Registered {len(self.hooks)} attention hooks")
    
    def _hook_attention_layers(self, transformer_name, transformer_module):
        """Hook individual attention layers within TwoWayTransformer"""
        
        # Hook self-attention
        if hasattr(transformer_module, 'self_attn'):
            hook_name = f"{transformer_name}_self_attn"
            hook = self._create_attention_hook(hook_name, transformer_module.self_attn)
            self.hooks.append(hook)
            print(f"  Hooked self-attention: {hook_name}")
        
        # Hook cross-attention token-to-image
        if hasattr(transformer_module, 'cross_attn_token_to_image'):
            hook_name = f"{transformer_name}_token_to_image"
            hook = self._create_attention_hook(hook_name, transformer_module.cross_attn_token_to_image)
            self.hooks.append(hook)
            print(f"  Hooked token-to-image attention: {hook_name}")
        
        # Hook cross-attention image-to-token
        if hasattr(transformer_module, 'cross_attn_image_to_token'):
            hook_name = f"{transformer_name}_image_to_token"
            hook = self._create_attention_hook(hook_name, transformer_module.cross_attn_image_to_token)
            self.hooks.append(hook)
            print(f"  Hooked image-to-token attention: {hook_name}")
    
    def _create_attention_hook(self, hook_name, attention_module):
        """Create a hook that captures attention weights"""
        
        # Store the original forward method
        original_forward = attention_module.forward
        
        def hooked_forward(x, xa=None, mask=None):
            # Call original forward but capture attention weights
            if hasattr(attention_module, 'attention_weights'):
                # Some attention modules store weights
                result = original_forward(x, xa, mask)
                self.attention_weights[hook_name] = attention_module.attention_weights.detach().cpu()
            else:
                # For modules that don't store weights, we need to modify the forward pass
                result = self._forward_with_attention_capture(attention_module, original_forward, x, xa, mask)
                if result is not None:
                    self.attention_weights[hook_name] = result[1].detach().cpu()
                    result = result[0]
                else:
                    result = original_forward(x, xa, mask)
            
            return result
        
        # Replace the forward method
        attention_module.forward = hooked_forward
        
        # Return a cleanup function
        def cleanup():
            attention_module.forward = original_forward
        
        return cleanup
    
    def _forward_with_attention_capture(self, attention_module, original_forward, x, xa, mask):
        """Modified forward pass that captures attention weights"""
        try:
            # This is a simplified approach - in practice, you might need to 
            # modify the actual attention implementation to return weights
            
            # For MultiheadAttention, we can try to capture during forward
            if hasattr(attention_module, 'multihead_attn'):
                # Store the original forward
                orig_multihead_forward = attention_module.multihead_attn.forward
                
                def capture_multihead_forward(query, key, value, *args, **kwargs):
                    # Call original forward
                    result = orig_multihead_forward(query, key, value, *args, **kwargs)
                    
                    # Try to compute attention weights
                    if hasattr(attention_module.multihead_attn, 'compute_attention_weights'):
                        attn_weights = attention_module.multihead_attn.compute_attention_weights(query, key)
                        return result, attn_weights
                    
                    return result, None
                
                attention_module.multihead_attn.forward = capture_multihead_forward
                result = original_forward(x, xa, mask)
                attention_module.multihead_attn.forward = orig_multihead_forward
                
                return result
            else:
                return None
                
        except Exception as e:
            print(f"Could not capture attention for {attention_module}: {e}")
            return None
    
    def clear_hooks(self):
        """Remove all hooks"""
        for cleanup_func in self.hooks:
            cleanup_func()
        self.hooks = []
        self.attention_weights = {}
        print("Enhanced TwoWayTransformer hooks cleared.")

print("Enhanced TwoWayTransformer Hook class loaded!")


Enhanced TwoWayTransformer Hook class loaded!


In [12]:
# Attention Map Visualization System
class AttentionMapVisualizer:
    def __init__(self):
        self.attention_maps = {}
        self.hooks = []
        self.image_features = None
        self.prompt_features = None
        self.enhanced_hook = EnhancedTwoWayTransformerHook()
        
    def register_attention_hooks(self, model):
        """Register enhanced attention hooks"""
        self.enhanced_hook.hook_two_way_transformer(model)
        
    def capture_features(self, model, image_path, example_idx=1):
        """Capture features with enhanced attention hooking"""
        print("Capturing features with enhanced attention hooks...")
        
        # Load image
        image = cv2.imread(f'{image_path}/example{example_idx}.png')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Configure prompts based on example
        if example_idx == 0:
            input_box = np.array([[4, 13, 1007, 1023]])
            input_point, input_label = None, None
            hq_token_only = False
        elif example_idx == 1:
            input_box = np.array([[306, 132, 925, 893]])
            input_point, input_label = None, None
            hq_token_only = True
        elif example_idx == 2:
            input_point = np.array([[495, 518], [217, 140]])
            input_label = np.ones(input_point.shape[0])
            input_box = None
            hq_token_only = True
        else:
            input_box = np.array([[306, 132, 925, 893]])
            input_point, input_label = None, None
            hq_token_only = True
        
        # Create predictor and run inference
        predictor = SamPredictor(model)
        predictor.set_image(image)
        
        # Store the image for visualization
        self.original_image = image
        self.input_box = input_box
        self.input_point = input_point
        self.input_label = input_label
        
        try:
            masks, scores, logits = predictor.predict(
                point_coords=input_point,
                point_labels=input_label,
                box=input_box,
                multimask_output=False,
                hq_token_only=hq_token_only,
            )
        except TypeError as e:
            if "hq_token_only" in str(e):
                masks, scores, logits = predictor.predict(
                    point_coords=input_point,
                    point_labels=input_label,
                    box=input_box,
                    multimask_output=False,
                )
            else:
                raise e
        
        self.masks = masks
        self.scores = scores
        
        # Copy captured attention weights
        self.attention_maps = self.enhanced_hook.attention_weights.copy()
        
        return masks, scores, logits
    
    def plot_attention_maps(self, save_path=None):
        """Plot attention maps between prompt and image features"""
        if not self.attention_maps:
            print("No attention maps captured. Run capture_features() first.")
            return
        
        print(f"Plotting {len(self.attention_maps)} attention maps...")
        
        # Create subplots
        n_maps = len(self.attention_maps)
        cols = min(3, n_maps)
        rows = (n_maps + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
        if rows == 1 and cols == 1:
            axes = [axes]
        elif rows == 1:
            axes = axes
        else:
            axes = axes.flatten()
        
        for i, (map_name, attention_map) in enumerate(self.attention_maps.items()):
            if i >= len(axes):
                break
                
            ax = axes[i]
            
            # Handle different attention map shapes
            if attention_map.dim() == 4:  # [batch, heads, seq_len, seq_len]
                # Average over heads and batch
                attn = attention_map.mean(dim=(0, 1)).numpy()
            elif attention_map.dim() == 3:  # [batch, seq_len, seq_len]
                # Average over batch
                attn = attention_map.mean(dim=0).numpy()
            else:
                attn = attention_map.numpy()
            
            # Plot attention map
            im = ax.imshow(attn, cmap='Blues', aspect='auto')
            ax.set_title(f'{map_name}\nShape: {attention_map.shape}', fontsize=10)
            ax.set_xlabel('Key Position')
            ax.set_ylabel('Query Position')
            
            # Add colorbar
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        
        # Hide unused subplots
        for i in range(len(self.attention_maps), len(axes)):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Attention maps saved to {save_path}")
        
        plt.show()
    
    def plot_attention_on_image(self, save_path=None):
        """Plot attention maps overlaid on the original image"""
        if not self.attention_maps or not hasattr(self, 'original_image'):
            print("No attention maps or image available. Run capture_features() first.")
            return
        
        print("Plotting attention maps overlaid on image...")
        
        # Create subplots
        n_maps = len(self.attention_maps)
        cols = min(2, n_maps)
        rows = (n_maps + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=(8*cols, 6*rows))
        if rows == 1 and cols == 1:
            axes = [axes]
        elif rows == 1:
            axes = axes
        else:
            axes = axes.flatten()
        
        for i, (map_name, attention_map) in enumerate(self.attention_maps.items()):
            if i >= len(axes):
                break
                
            ax = axes[i]
            
            # Handle attention map shapes
            if attention_map.dim() == 4:
                attn = attention_map.mean(dim=(0, 1)).numpy()
            elif attention_map.dim() == 3:
                attn = attention_map.mean(dim=0).numpy()
            else:
                attn = attention_map.numpy()
            
            # Resize attention map to match image
            image_h, image_w = self.original_image.shape[:2]
            attn_resized = cv2.resize(attn, (image_w, image_h))
            
            # Show original image
            ax.imshow(self.original_image)
            
            # Overlay attention map
            ax.imshow(attn_resized, alpha=0.6, cmap='hot')
            
            # Add prompt indicators
            if self.input_box is not None:
                box = self.input_box[0]
                x0, y0 = box[0], box[1]
                w, h = box[2] - box[0], box[3] - box[1]
                rect = plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor='none', linewidth=2)
                ax.add_patch(rect)
            
            if self.input_point is not None and self.input_label is not None:
                show_points(self.input_point, self.input_label, ax)
            
            ax.set_title(f'{map_name}\nAttention Overlay', fontsize=12)
            ax.axis('off')
        
        # Hide unused subplots
        for i in range(len(self.attention_maps), len(axes)):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Attention overlays saved to {save_path}")
        
        plt.show()
    
    def plot_prompt_image_attention_flow(self, save_path=None):
        """Create a comprehensive visualization showing the attention flow between prompts and image"""
        if not self.attention_maps or not hasattr(self, 'original_image'):
            print("No attention maps or image available. Run capture_features() first.")
            return
        
        print("Creating comprehensive attention flow visualization...")
        
        # Create figure with multiple subplots
        fig = plt.figure(figsize=(20, 12))
        
        # Original image with prompts
        ax1 = plt.subplot(2, 4, 1)
        ax1.imshow(self.original_image)
        if self.input_box is not None:
            box = self.input_box[0]
            x0, y0 = box[0], box[1]
            w, h = box[2] - box[0], box[3] - box[1]
            rect = plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor='none', linewidth=3)
            ax1.add_patch(rect)
            ax1.text(x0, y0-10, 'Prompt Box', color='green', fontsize=12, fontweight='bold')
        
        if self.input_point is not None and self.input_label is not None:
            show_points(self.input_point, self.input_label, ax1, marker_size=300)
            ax1.text(10, 30, 'Prompt Points', color='white', fontsize=12, fontweight='bold',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor='black', alpha=0.7))
        
        ax1.set_title('Input Image with Prompts', fontsize=14, fontweight='bold')
        ax1.axis('off')
        
        # Predicted mask
        ax2 = plt.subplot(2, 4, 2)
        ax2.imshow(self.original_image)
        if len(self.masks) > 0:
            show_mask_image(self.masks[0], ax2, random_color=False, borders=True)
        ax2.set_title(f'Predicted Mask\nScore: {self.scores[0]:.4f}', fontsize=14, fontweight='bold')
        ax2.axis('off')
        
        # Attention maps
        attention_positions = [(2, 4, 3), (2, 4, 4), (2, 4, 5), (2, 4, 6), (2, 4, 7), (2, 4, 8)]
        
        for i, (map_name, attention_map) in enumerate(self.attention_maps.items()):
            if i >= len(attention_positions):
                break
            
            pos = attention_positions[i]
            ax = plt.subplot(pos[0], pos[1], pos[2])
            
            # Process attention map
            if attention_map.dim() == 4:
                attn = attention_map.mean(dim=(0, 1)).numpy()
            elif attention_map.dim() == 3:
                attn = attention_map.mean(dim=0).numpy()
            else:
                attn = attention_map.numpy()
            
            # Plot attention map
            im = ax.imshow(attn, cmap='viridis', aspect='auto')
            ax.set_title(f'{map_name}\n{attention_map.shape}', fontsize=10)
            
            # Add colorbar
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        
        # Hide unused subplots
        for i in range(len(self.attention_maps), len(attention_positions)):
            pos = attention_positions[i]
            ax = plt.subplot(pos[0], pos[1], pos[2])
            ax.set_visible(False)
        
        plt.suptitle('TwoWayTransformer Attention Analysis\nPrompt-Image Interaction', 
                    fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Comprehensive attention flow saved to {save_path}")
        
        plt.show()
    
    def analyze_attention_statistics(self):
        """Analyze and print statistics about the captured attention maps"""
        if not self.attention_maps:
            print("No attention maps to analyze.")
            return
        
        print("="*80)
        print("ATTENTION MAP STATISTICS")
        print("="*80)
        
        for map_name, attention_map in self.attention_maps.items():
            print(f"\n{map_name}:")
            print(f"  Shape: {attention_map.shape}")
            print(f"  Min value: {attention_map.min().item():.6f}")
            print(f"  Max value: {attention_map.max().item():.6f}")
            print(f"  Mean value: {attention_map.mean().item():.6f}")
            print(f"  Std value: {attention_map.std().item():.6f}")
            
            # Analyze attention patterns
            if attention_map.dim() >= 2:
                # Find most attended positions
                if attention_map.dim() == 4:  # [batch, heads, seq_len, seq_len]
                    avg_attn = attention_map.mean(dim=(0, 1))
                elif attention_map.dim() == 3:  # [batch, seq_len, seq_len]
                    avg_attn = attention_map.mean(dim=0)
                else:
                    avg_attn = attention_map
                
                # Find top attended positions
                flat_attn = avg_attn.flatten()
                top_indices = torch.topk(flat_attn, k=min(5, len(flat_attn))).indices
                
                print(f"  Top attended positions: {top_indices.tolist()}")
                
                # Analyze attention sparsity
                threshold = 0.1 * avg_attn.max()
                sparse_ratio = (avg_attn < threshold).float().mean().item()
                print(f"  Attention sparsity (below 10% max): {sparse_ratio:.3f}")
    
    def clear_hooks(self):
        """Clear both regular and enhanced hooks"""
        self.enhanced_hook.clear_hooks()

print("Attention Map Visualizer class loaded!")


Attention Map Visualizer class loaded!


In [13]:
# Main Function to Visualize TwoWayTransformer Attention Maps
def visualize_two_way_transformer_attention(checkpoint_path=None, model_type='vit_l', example_idx=1, image_path='./input_imgs/'):
    """
    Main function to visualize attention maps between prompt and image in TwoWayTransformer
    
    Args:
        checkpoint_path: Path to SAM model checkpoint
        model_type: Type of SAM model ('vit_l', 'vit_h', etc.)
        example_idx: Which example image to test
        image_path: Path to input images
    """
    
    print("="*80)
    print("TWO-WAY TRANSFORMER ATTENTION MAP VISUALIZATION")
    print("="*80)
    print(f"Model: {model_type}")
    print(f"Example: {example_idx}")
    print(f"Image path: {image_path}")
    
    # Default checkpoint path if not provided
    if checkpoint_path is None:
        checkpoint_path = '/u/ctran3/Sam_quantization/pretrained_checkpoint/sam_hq_vit_l.pth'
    
    try:
        # Load SAM model
        print("\nLoading SAM model...")
        model = sam_hq_model_registry[model_type](checkpoint=checkpoint_path).to('cuda')
        model.eval()
        print("Model loaded successfully!")
        
        # Create enhanced attention visualizer
        print("\nInitializing attention visualizer...")
        visualizer = AttentionMapVisualizer()
        
        # Register attention hooks
        print("\nRegistering attention hooks...")
        visualizer.register_attention_hooks(model)
        
        # Capture features and attention maps
        print("\nRunning inference to capture attention maps...")
        masks, scores, logits = visualizer.capture_features(model, image_path, example_idx)
        
        print(f"Inference completed!")
        print(f"Predicted mask score: {scores[0]:.4f}")
        print(f"Captured {len(visualizer.attention_maps)} attention maps")
        
        # Analyze attention statistics
        print("\nAnalyzing attention statistics...")
        visualizer.analyze_attention_statistics()
        
        # Create visualizations
        print("\nCreating attention visualizations...")
        
        # 1. Basic attention maps
        print("1. Plotting basic attention maps...")
        visualizer.plot_attention_maps(save_path=f'two_way_attention_maps_{model_type}_{example_idx}.png')
        
        # 2. Attention overlays on image
        print("2. Plotting attention overlays on image...")
        visualizer.plot_attention_on_image(save_path=f'two_way_attention_overlays_{model_type}_{example_idx}.png')
        
        # 3. Comprehensive attention flow
        print("3. Creating comprehensive attention flow visualization...")
        visualizer.plot_prompt_image_attention_flow(save_path=f'two_way_attention_flow_{model_type}_{example_idx}.png')
        
        # Summary
        print("\n" + "="*80)
        print("ATTENTION VISUALIZATION SUMMARY")
        print("="*80)
        print(f"Model: {model_type}")
        print(f"Example: {example_idx}")
        print(f"Mask score: {scores[0]:.4f}")
        print(f"Attention maps captured: {len(visualizer.attention_maps)}")
        
        if visualizer.attention_maps:
            print("\nCaptured attention maps:")
            for map_name, attn_map in visualizer.attention_maps.items():
                print(f"  {map_name}: {attn_map.shape}")
        
        print("\nVisualizations created:")
        print(f"  - Basic attention maps: two_way_attention_maps_{model_type}_{example_idx}.png")
        print(f"  - Attention overlays: two_way_attention_overlays_{model_type}_{example_idx}.png")
        print(f"  - Comprehensive flow: two_way_attention_flow_{model_type}_{example_idx}.png")
        
        return visualizer
        
    except Exception as e:
        print(f"Error during attention visualization: {str(e)}")
        import traceback
        traceback.print_exc()
        return None
        
    finally:
        # Cleanup
        if 'visualizer' in locals():
            visualizer.clear_hooks()
        print("\nAttention visualization completed!")

def compare_attention_across_examples(model_type='vit_l', examples=[0, 1, 2], image_path='./input_imgs/'):
    """
    Compare attention patterns across different examples
    
    Args:
        model_type: Type of SAM model
        examples: List of example indices to compare
        image_path: Path to input images
    """
    
    print("="*80)
    print("COMPARING ATTENTION PATTERNS ACROSS EXAMPLES")
    print("="*80)
    
    results = {}
    
    for example_idx in examples:
        print(f"\n{'='*60}")
        print(f"Analyzing Example {example_idx}")
        print(f"{'='*60}")
        
        visualizer = visualize_two_way_transformer_attention(
            model_type=model_type,
            example_idx=example_idx,
            image_path=image_path
        )
        
        if visualizer:
            results[example_idx] = {
                'visualizer': visualizer,
                'attention_maps': visualizer.attention_maps,
                'score': visualizer.scores[0] if hasattr(visualizer, 'scores') else 0
            }
    
    # Summary comparison
    if results:
        print(f"\n{'='*80}")
        print("ATTENTION PATTERN COMPARISON SUMMARY")
        print(f"{'='*80}")
        
        for example_idx, result in results.items():
            print(f"\nExample {example_idx}:")
            print(f"  Score: {result['score']:.4f}")
            print(f"  Attention maps: {len(result['attention_maps'])}")
            
            for map_name, attn_map in result['attention_maps'].items():
                print(f"    {map_name}: {attn_map.shape}")
    
    return results

print("Main visualization functions loaded!")


Main visualization functions loaded!


In [14]:
# Run TwoWayTransformer Attention Map Visualization
print("Starting TwoWayTransformer Attention Map Analysis...")
print("This will visualize the attention between prompts and image features in the MaskDecoder")
print("\nThe visualization will show:")
print("1. Basic attention maps between prompt and image tokens")
print("2. Attention overlays on the original image")
print("3. Comprehensive attention flow visualization")
print("4. Statistical analysis of attention patterns")

# Run the attention visualization for example 1
attention_visualizer = visualize_two_way_transformer_attention(
    model_type='vit_l',      # Use ViT-L model
    example_idx=1,          # Test on example 1 (box prompt)
    image_path='./input_imgs/'  # Path to your input images
)

print("\nTwoWayTransformer attention visualization completed!")
print("The analysis shows how the model attends to different parts of the image based on the prompts.")
print("This helps understand the prompt-image interaction mechanism in SAM's MaskDecoder.")


Starting TwoWayTransformer Attention Map Analysis...
This will visualize the attention between prompts and image features in the MaskDecoder

The visualization will show:
1. Basic attention maps between prompt and image tokens
2. Attention overlays on the original image
3. Comprehensive attention flow visualization
4. Statistical analysis of attention patterns
TWO-WAY TRANSFORMER ATTENTION MAP VISUALIZATION
Model: vit_l
Example: 1
Image path: ./input_imgs/

Loading SAM model...
Error during attention visualization: name 'sam_hq_model_registry' is not defined

Attention visualization completed!

TwoWayTransformer attention visualization completed!
The analysis shows how the model attends to different parts of the image based on the prompts.
This helps understand the prompt-image interaction mechanism in SAM's MaskDecoder.


Traceback (most recent call last):
  File "/tmp/ipykernel_933924/1871066213.py", line 27, in visualize_two_way_transformer_attention
    model = sam_hq_model_registry[model_type](checkpoint=checkpoint_path).to('cuda')
NameError: name 'sam_hq_model_registry' is not defined


In [15]:
# Optional: Compare attention patterns across different examples
# Uncomment the lines below to analyze multiple examples

# print("\n" + "="*80)
# print("COMPARING ATTENTION PATTERNS ACROSS EXAMPLES")
# print("="*80)

# attention_comparison = compare_attention_across_examples(
#     model_type='vit_l',
#     examples=[0, 1, 2],  # Compare examples 0, 1, and 2
#     image_path='./input_imgs/'
# )

print("\n" + "="*80)
print("ATTENTION VISUALIZATION COMPLETED!")
print("="*80)
print("\nWhat you've learned:")
print("1. How TwoWayTransformer processes prompts and image features")
print("2. Which image regions receive the most attention")
print("3. How different attention layers (self, cross) behave")
print("4. The relationship between attention patterns and segmentation quality")
print("\nGenerated visualizations:")
print("- Basic attention maps: Show raw attention weights")
print("- Attention overlays: Show attention on the actual image")
print("- Comprehensive flow: Complete analysis with prompts and predictions")
print("\nThis analysis helps understand SAM's internal attention mechanisms!")



ATTENTION VISUALIZATION COMPLETED!

What you've learned:
1. How TwoWayTransformer processes prompts and image features
2. Which image regions receive the most attention
3. How different attention layers (self, cross) behave
4. The relationship between attention patterns and segmentation quality

Generated visualizations:
- Basic attention maps: Show raw attention weights
- Attention overlays: Show attention on the actual image
- Comprehensive flow: Complete analysis with prompts and predictions

This analysis helps understand SAM's internal attention mechanisms!
